# Risk Manager

## Overview

This notebook demonstrates `RiskManager.validate_order` in `risk_manager` using compact proposed buy orders.
- Problem: a signal can request more cash, position exposure, or order quantity than the live-trading limits allow.
- Approach: validate representative orders against the configured maximum order size, maximum position, available cash, and current position.
- Risk Validation: It compares accepted and rejected order proposals without creating an exchange order.


In [ ]:
import pandas as pd
from IPython.display import display


In [ ]:
from src.live_trading.config import load_live_trading_config
from src.live_trading.risk_manager import RiskManager


## Risk Validation

This cell defines a compact set of proposed buy orders.
- `amount` is the base-asset quantity to buy.
- `price` converts the proposed amount into quote-currency cost.
- `current_position` and `available_cash` represent the state available before the order.


In [ ]:
config = load_live_trading_config()
risk_manager = RiskManager(config.max_order_size, config.max_position)

proposals = pd.DataFrame(
    [
        {
            "label": "within limits",
            "side": "buy",
            "amount": config.order_size,
            "price": 100.0,
            "current_position": 0.0,
            "available_cash": 1_000.0,
        },
        {
            "label": "exceeds order size",
            "side": "buy",
            "amount": config.max_order_size * 1.1,
            "price": 100.0,
            "current_position": 0.0,
            "available_cash": 1_000.0,
        },
        {
            "label": "exceeds position",
            "side": "buy",
            "amount": config.order_size,
            "price": 100.0,
            "current_position": config.max_position,
            "available_cash": 1_000.0,
        },
        {
            "label": "exceeds cash",
            "side": "buy",
            "amount": config.order_size,
            "price": 100.0,
            "current_position": 0.0,
            "available_cash": 0.01,
        },
    ]
)

validation_results = []
for proposal in proposals.to_dict("records"):
    try:
        risk_manager.validate_order(
            side=proposal["side"],
            amount=proposal["amount"],
            price=proposal["price"],
            current_position=proposal["current_position"],
            available_cash=proposal["available_cash"],
        )
        validation_results.append({"label": proposal["label"], "result": "accepted"})
    except ValueError as error:
        validation_results.append({"label": proposal["label"], "result": str(error)})

display(pd.DataFrame(validation_results))
